# 04 Schema Enforcement

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
See how Delta <b>rejects</b> writes that don't match the table schema (extra columns, wrong types, nulls in <code>NOT NULL</code> columns), what it accepts (missing nullable columns, safe type upcasts), and how that compares with writing plain Parquet files.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Schema enforcement (also called schema validation) is Delta's first line of defence against bad data: a source that suddenly sends <code>amount</code> as text, or adds 5 unexpected columns, fails loudly instead of silently corrupting a table that dashboards depend on. The exam asks what happens when a write doesn't match the schema, and how to allow intended changes (lesson 05).
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The rules on write</b><br>
| Incoming data vs table | Result |<br>|---|---|<br>| Same columns and types | ✅ Written |<br>| <b>Missing</b> column that is nullable | ✅ Written, the column is null |<br>| <b>Extra</b> column not in the table | ❌ <code>DELTA_METADATA_MISMATCH</code> / schema mismatch error (unless schema evolution is enabled) |<br>| <b>Different type</b> that can't be safely cast (string → int) | ❌ Error |<br>| Narrower numeric type (<code>int</code> into a <code>bigint</code> column) | DataFrame append: ❌ rejected, so cast first. SQL <code>INSERT</code>: ✅ cast by standard ANSI assignment rules |<br>| <b>Null</b> in a <code>NOT NULL</code> column | ❌ <code>DELTA_NOT_NULL_CONSTRAINT_VIOLATED</code> |<br>| Column names differing only by case | Treated as the same column (Delta is case-insensitive but case-preserving) |<br><br>
The check runs <b>before</b> commit, so a rejected write leaves the table unchanged.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A payments table had been loaded daily for two years. One morning a partner changed <code>amount</code> from a number to a string like <code>"120.50 USD"</code>. With plain Parquet files, the new files landed next to the old ones and every downstream query failed with type errors days later. With Delta, the 06:00 load failed immediately with a clear schema mismatch error, the on-call engineer was alerted, and the table stayed correct.
</div>

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a <code>payments</code> table with a strict schema, including a <code>NOT NULL</code> key.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The table schema: <code>payment_id</code> bigint not null, <code>customer</code> string, <code>amount</code> double, <code>paid_on</code> date.
</div>

In [0]:
DB = "workspace.delta_lake"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {DB}.files")
BASE = "/Volumes/workspace/delta_lake/files/lesson04"

In [0]:
from datetime import date
from pyspark.sql import functions as F

table = f"{DB}.payments"
spark.sql(f"""
    CREATE OR REPLACE TABLE {table} (
        payment_id BIGINT NOT NULL,
        customer   STRING,
        amount     DOUBLE,
        paid_on    DATE
    )
""")
spark.sql(f"INSERT INTO {table} VALUES (1, 'Amal', 120.5, DATE'2024-11-01'), (2, 'John', 80.0, DATE'2024-11-02')")
spark.table(table).printSchema()

def try_append(label, df):
    """Append df to the table and report success or the error, without stopping the notebook."""
    before = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()["version"]
    try:
        df.write.mode("append").saveAsTable(table)
        print(f"✅ {label}: written")
    except Exception as e:
        cond = getattr(e, "getCondition", lambda: None)() or type(e).__name__
        print(f"❌ {label}: rejected ({cond})")
    after = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()["version"]
    print(f"   table version {before} -> {after}")

## 1. An extra column is rejected

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Appends rows that contain a <code>currency</code> column the table doesn't have.<br><br>
<b>Why it matters</b><br>An unexpected column usually means the source changed. Delta stops the write so a human can decide: evolve the table (lesson 05) or drop the column.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>❌ rejected</code> with a schema mismatch error condition, and the table version doesn't change.
</div>

In [0]:
extra_col = spark.createDataFrame(
    [(3, "Vivek", 300.0, date(2024, 11, 3), "AED")],
    "payment_id BIGINT, customer STRING, amount DOUBLE, paid_on DATE, currency STRING",
)
try_append("extra column 'currency'", extra_col)

## 2. A wrong type is rejected

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Appends rows where <code>amount</code> is a string.<br><br>
<b>Why it matters</b><br>Delta doesn't silently convert text into numbers. That would turn <code>"120.50 USD"</code> into null, or fail later. It rejects the write, so you fix the type in your code with <code>cast</code>/<code>try_cast</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>❌ rejected</code>, and the table version is unchanged.
</div>

In [0]:
wrong_type = spark.createDataFrame(
    [(4, "Sara", "120.50 USD", date(2024, 11, 4))],
    "payment_id BIGINT, customer STRING, amount STRING, paid_on DATE",
)
try_append("amount as STRING", wrong_type)

## 3. Nulls in a NOT NULL column are rejected

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Appends a row with a null <code>payment_id</code>.<br><br>
<b>Why it matters</b><br><code>NOT NULL</code> is enforced on every write by every writer. Keys can't silently become null.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>❌ rejected</code> with <code>DELTA_NOT_NULL_CONSTRAINT_VIOLATED</code> (or a similar condition), and the version is unchanged.
</div>

In [0]:
null_key = spark.createDataFrame([(None, "Ghost", 10.0, date(2024, 11, 5))],
                                 "payment_id BIGINT, customer STRING, amount DOUBLE, paid_on DATE")
try_append("null payment_id", null_key)

## 4. What IS accepted: missing columns, and casts done for you by SQL

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Appends rows without the <code>customer</code> column. Then appends rows where <code>payment_id</code> is an <code>int</code> and <code>amount</code> a <code>float</code>: first as a DataFrame, then with SQL <code>INSERT</code>.<br><br>
<b>Why it matters</b><br>A missing nullable column is safe: it becomes null. Type differences are stricter for DataFrame writes: Delta wants the exact table types, so you <code>cast</code> in your code. SQL <code>INSERT</code> applies the standard ANSI assignment rules, which cast an <code>INT</code> into a <code>BIGINT</code> column for you.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The missing-column append succeeds and shows a null customer. The DataFrame append with <code>int</code>/<code>float</code> is rejected (<code>DELTA_FAILED_TO_MERGE_FIELDS</code>). The same values succeed through SQL <code>INSERT</code>, and after casting in the DataFrame. The table types are unchanged.
</div>

In [0]:
missing_col = spark.createDataFrame([(5, 42.0, date(2024, 11, 6))], "payment_id BIGINT, amount DOUBLE, paid_on DATE")
try_append("missing nullable column 'customer'", missing_col)

narrower = spark.createDataFrame([(6, "Priya", 99.5, date(2024, 11, 7))], "payment_id INT, customer STRING, amount FLOAT, paid_on DATE")
try_append("DataFrame with int and float", narrower)

# The same values through SQL: INSERT casts INT -> BIGINT and FLOAT -> DOUBLE by ANSI assignment rules
spark.sql(f"INSERT INTO {table} SELECT CAST(7 AS INT), 'Ravi', CAST(99.5 AS FLOAT), DATE'2024-11-07'")
print("✅ SQL INSERT with INT and FLOAT values: written")

# Or cast in the DataFrame to the table's types
try_append("DataFrame after casting", narrower.select(
    F.col("payment_id").cast("bigint"), "customer", F.col("amount").cast("double"), "paid_on"))

spark.table(table).orderBy("payment_id").show()
spark.table(table).printSchema()

## 5. SQL INSERT is enforced too

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Inserts through SQL with the wrong number of columns, then with a value that can't be cast.<br><br>
<b>Why it matters</b><br>Enforcement applies to every write path: DataFrame writers, SQL <code>INSERT</code>, <code>MERGE</code> and streaming.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Two rejected inserts with their error conditions.
</div>

In [0]:
for sql in [
    f"INSERT INTO {table} VALUES (7, 'Ravi', 10.0, DATE'2024-11-08', 'extra value')",
    f"INSERT INTO {table} VALUES (8, 'Meena', 'not a number', DATE'2024-11-09')",
]:
    try:
        spark.sql(sql)
        print("✅ inserted")
    except Exception as e:
        print("❌", getattr(e, "getCondition", lambda: type(e).__name__)())

## 6. Compare: plain Parquet has no enforcement

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes two batches with <b>different</b> schemas into the same Parquet folder, then reads it.<br><br>
<b>Why it matters</b><br>Parquet folders accept anything. The mismatch shows up only when someone reads: missing columns, columns silently dropped, or type errors. That's the "data swamp" problem Delta was built to solve.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Both writes succeed. Reading the folder silently ignores one batch's extra column (or fails on the conflicting type), depending on which file's schema Spark picks.
</div>

In [0]:
pq = f"{BASE}/payments_parquet"
spark.createDataFrame([(1, 120.5)], "payment_id BIGINT, amount DOUBLE").write.mode("overwrite").parquet(pq)
spark.createDataFrame([(2, "80 USD", "GBP")], "payment_id BIGINT, amount STRING, currency STRING").write.mode("append").parquet(pq)
print("both Parquet writes succeeded")
try:
    df = spark.read.parquet(pq)
    print("schema Spark chose:", df.dtypes)
    df.show()
except Exception as e:
    print("reading the folder failed:", type(e).__name__)

## Under the hood

```
write(df) to Delta table
   1. read the table's current schema from the log (metaData action)
   2. compare with df's schema:
        extra columns?       -> error (unless mergeSchema / autoMerge)
        incompatible types?  -> error
        missing nullable?    -> fill with null
        narrower numerics?   -> cast up
   3. write files with the TABLE's schema, check NOT NULL / CHECK constraints
   4. commit (or, if any check failed, never commit)
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> the check happens when the write runs, before the commit.

Enforcement compares **names**, not positions (except for SQL `INSERT` without a column list, which is positional). Column names are matched case-insensitively.

In [0]:
spark.sql(f"DESCRIBE HISTORY {table}").select("version", "operation", "operationMetrics.numOutputRows").show()

In [0]:
# Optional cleanup
spark.sql(f"DROP TABLE IF EXISTS {table}")
dbutils.fs.rm(BASE, True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>A schema mismatch detected when writing to the Delta table</code></b><br>
The DataFrame has extra or differently typed columns. Select and cast to the table schema, or enable schema evolution deliberately (lesson 05).<br><br>
**⛔ Problem: <code>DELTA_NOT_NULL_CONSTRAINT_VIOLATED</code>**<br>A required column contains nulls. Fix the source, filter, or quarantine those rows.<br><br>
<b>⛔ Problem: "it worked yesterday"</b><br>The source schema changed. Enforcement did its job, so compare the incoming schema with the table's.<br><br>
**⛔ Problem: positional <code>INSERT</code> puts values in the wrong columns**<br>List the columns explicitly in <code>INSERT INTO t (a, b, c)</code>.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is schema enforcement in Delta Lake?</b><br>
Delta validates every write against the table's schema and rejects writes with extra columns, incompatible types, or nulls in <code>NOT NULL</code> columns. The check happens before commit, so a bad write leaves the table unchanged.<br><br>

<b>🎤 2. What kinds of differences does Delta accept on write?</b><br>
Missing nullable columns, which are filled with null (or the column's default). DataFrame writes otherwise need matching types, while SQL <code>INSERT</code> casts values using ANSI assignment rules, for example <code>INT</code> into <code>BIGINT</code>. New columns and incompatible types are rejected.<br><br>

<b>🎤 3. Why is schema enforcement useful?</b><br>
It stops upstream changes from silently corrupting downstream tables, and it fails fast at load time with a clear error instead of breaking reports later.<br><br>

<b>🎤 4. How is this different from writing Parquet files?</b><br>
A Parquet folder accepts files with any schema. Problems only appear when reading, as dropped columns, nulls or type errors. Delta keeps one schema in the log and validates every write.<br><br>

<b>🎤 5. How do you allow intentional schema changes?</b><br>
With schema evolution: <code>mergeSchema</code> on an append, <code>overwriteSchema</code> on an overwrite, <code>WITH SCHEMA EVOLUTION</code> or auto-merge for <code>MERGE</code>, or explicit <code>ALTER TABLE ... ADD COLUMNS</code>.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A data engineer appends a DataFrame with one extra column to an existing Delta table, without any options. What happens?</b><br>
A. The extra column is silently dropped<br>
B. The write fails with a schema mismatch error and the table is unchanged<br>
C. The table schema is updated automatically<br>
D. The extra column is written to a separate file<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. An append is missing a nullable column that exists in the target Delta table. What happens?</b><br>
A. The write fails<br>
B. The write succeeds and the missing column is null for the new rows<br>
C. The column is dropped from the table<br>
D. Delta fills it with default zero values<br>
<details><summary><b>Show answer</b></summary><b>B.</b> If the column has a declared <code>DEFAULT</code>, that value is used instead.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a table <code>sensor_readings (sensor_id INT NOT NULL, reading DOUBLE, ts TIMESTAMP)</code></li><li>Try to append: (a) a row with an extra <code>unit</code> column, (b) <code>reading</code> as a string, (c) a null <code>sensor_id</code>, (d) a row without <code>ts</code>. Predict each result before running it</li><li>Fix (a) and (b) in your DataFrame code (drop the extra column, <code>try_cast</code> the string) so the append succeeds</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
from datetime import datetime

# 1. Strict table
table = f"{DB}.sensor_readings"
spark.sql(f"CREATE OR REPLACE TABLE {table} (sensor_id INT NOT NULL, reading DOUBLE, ts TIMESTAMP)")
spark.sql(f"INSERT INTO {table} VALUES (1, 20.5, TIMESTAMP'2024-11-01 10:00:00')")

# 2. Predictions: (a) rejected, (b) rejected, (c) rejected, (d) accepted with null ts
now = datetime(2024, 11, 1, 11, 0)
try_append("(a) extra column", spark.createDataFrame([(2, 21.0, now, "C")], "sensor_id INT, reading DOUBLE, ts TIMESTAMP, unit STRING"))
try_append("(b) string reading", spark.createDataFrame([(3, "22.1", now)], "sensor_id INT, reading STRING, ts TIMESTAMP"))
try_append("(c) null sensor_id", spark.createDataFrame([(None, 22.1, now)], "sensor_id INT, reading DOUBLE, ts TIMESTAMP"))
try_append("(d) missing ts", spark.createDataFrame([(4, 19.9)], "sensor_id INT, reading DOUBLE"))

# 3. Conform the data to the table instead of loosening the table
fixed = (
    spark.createDataFrame([(5, "23.4", now, "C")], "sensor_id INT, reading STRING, ts TIMESTAMP, unit STRING")
    .drop("unit")
    .withColumn("reading", F.col("reading").try_cast("double"))
)
try_append("fixed row", fixed)
spark.table(table).orderBy("sensor_id").show()
spark.sql(f"DROP TABLE {table}")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Delta validates every write against the table schema before committing</li><li>Rejected: extra columns, incompatible types, nulls in <code>NOT NULL</code> columns</li><li>Accepted: missing nullable columns (null or <code>DEFAULT</code>). SQL <code>INSERT</code> casts compatible types, while DataFrame appends need exact types</li><li>Applies to DataFrame writes, SQL <code>INSERT</code>, <code>MERGE</code> and streaming</li><li>Plain Parquet folders have no enforcement</li><li>To change the schema on purpose, use schema evolution (lesson 05)</li></ul>
</div>

| Situation | What to do |
|---|---|
| Extra column, unintended | `df.drop("col")` or `df.select(table_columns)` |
| Wrong type | `F.col("c").try_cast("double")` |
| Nulls in a key | Filter, or quarantine the rows |
| Intended new column | `mergeSchema` / `ALTER TABLE ADD COLUMNS` (lesson 05) |

## Git commit

```
git add 03_delta_lake/04_schema_enforcement.ipynb
git commit -m "add 03_04 schema enforcement notebook"
```